In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
import os
import json
import re

In [19]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LORA_PATH = "./qwen34_lora_adapter"

In [20]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",          # <-- IMPORTANT
    trust_remote_code=True
)

# Resize embeddings after pad token
model.resize_token_embeddings(len(tokenizer))

# Load LoRA adapter (not merged)
model = PeftModel.from_pretrained(
    model,
    LORA_PATH,
    device_map="cpu",          # <-- match base model
)

model.eval()

Loading checkpoint shards: 100%|██████████| 3/3 [00:08<00:00,  2.77s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151669, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, 

In [21]:
file = "../Knowledge-graph/doc_146120936/questions_146120936.txt"

all_questions = []

with open(file, "r", encoding="utf-8") as f:
        qs = [q.strip() for q in f.readlines() if q.strip()]
        all_questions.extend(qs)

In [ ]:
print(f"Total questions found: {len(all_questions)}\n")

doc_folder = os.path.dirname(file)  # ex: ../Knowledge-graph/doc_146120936
doc_name = os.path.basename(doc_folder)  # ex: doc_146120936
output_file = os.path.join(doc_folder, f"LoRA-Output-4B_{doc_name}.json")

results = {"results": []}

print(f"\nProcessing file: {file}\n")


Total questions found: 54


Processing file: ../Knowledge-graph/doc_146120936/questions_146120936.txt



In [23]:
# Progress bar
pbar = tqdm(total=len(all_questions), desc="Processing questions")

results = {"results": []}

with open(file, "r", encoding="utf-8") as f:
      questions = f.readlines()

for question in questions:

    if question.strip() == "":
        continue

    pbar.update(1)

    question = question.strip()

    print(f"Question: {question}\n")

    prompt = f"""
You are an information extraction system.

TASK:
Given a research question, output factual triplets in JSON.

Rules:
1. Triplet format: [subject:ENTITY_LABEL, RELATION_LABEL, object:ENTITY_LABEL]
2. ENTITY_LABEL must be one of: Method, Task, Dataset
3. RELATION_LABEL must be one of:
   Used-For, Part-Of, Compare-With, SubClass-Of, Synonym-Of,
   Evaluated-With, Benchmark-For, Trained-With, SubTask-Of
4. Output ONLY valid JSON with a single field "triplets".
5. "triplets" must be an array of strings in the exact format above.
6. Do not include any text outside the JSON.

QUESTION:
{question}

OUTPUT FORMAT:
{{
  "triplets": [
    "[subject:LABEL, RELATION_LABEL, object:LABEL]"
  ]
}}
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,         # allow stochastic but controlled generation
            eos_token_id = tokenizer.eos_token_id,  # stop at JSON closing
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print("Raw Assistant Response:\n", response)
    
    match = re.search(r'\{.*\}', response, re.DOTALL)
    if match:
        try:
            triplets_json = json.loads(match.group(0))
        except:
            triplets_json = {"triplets": [], "raw_output": response}
    else:
        triplets_json = {"triplets": [], "raw_output": response}


    results["results"].append({
        "question": question,
        "triplets": triplets_json
        })

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"\nSaved output to: {output_file}\n")

pbar.close()

Processing questions:   2%|▏         | 1/54 [00:15<13:57, 15.81s/it]


Question: What is a subtask of dense prediction tasks?



Raw Assistant Response:
 {
  "triplets": [
    "[subject:Dense-Pixel-Labeling, RELATION_LABEL, object:Dense-Prediction-Tasks]"
  ]
} We are given the research question: "What is a subtask of dense prediction tasks?"

Step 1: Identify the subject and object based on the question.
- The question is asking for a subtask of dense prediction tasks.
- So, the subject should be a subtask, and the object should be "Dense-Prediction-Tasks".

Step 2: Determine the correct RELATION_LABEL.
- The relation "SubTask-Of" fits the context because we are looking for a subtask of dense prediction tasks.

Step 3: Identify the correct ENTITY_LABEL.
- The subject (a subtask) must be labeled as a Task, since it is a type of task.
- The object (dense prediction tasks) must also be labeled as a Task.

Step 4: Construct the triplet.
- [subject:Task, SubTask-Of, object:Task]

Step 5: Ensure the format matches the required JSON structure.

Final Output:
{
  "triplets": [
    "[subject:Task, SubTask-Of, object:Tas

Raw Assistant Response:
 {
  "triplets": [
    "[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]"
  ]
}

Saved output to: ../Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json

Question: What method is used for Feature Pyramid Network?



Raw Assistant Response:
 {
  "triplets": [
    "[subject:Feature Pyramid Network, Used-For, Task:Object Detection]"
  ]
}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Object Detection]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Instance Segmentation]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Semantic Segmentation]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Segmentation]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Classification]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Super Resolution]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Inpainting]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Dehazing]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Denoising]"]}{"triplets": ["[subject:Feature Pyramid Network, Used-For, Task:Image Style Transfer

Saved output to: .

...

Raw Assistant Response:
 {
  "triplets": [
    "[subject:Global&Local, Part-Of, Method]"
  ]
} {"triplets": ["[subject:Global&Local, Part-Of, Method]"]}{"triplets": ["[subject:Global&Local, Part-Of, Method]"]}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets": ["[subject:Global&Local, Part-Of, Method]"}{"triplets":

Saved output to: ../Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json

Question: Which method is part of Partial Conv?



Raw Assistant Response:
 {
  "triplets": [
    "[subject:Partial Conv, Part-Of, Task:Segmentation]"
  ]
} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]"]} {"triplets": ["[subject:Partial Conv, Part-Of, Task:Segmentation]

Saved output to: ../Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json

Question: Which method is part of Mask R-CNN?



Raw Assistant Response:
 {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets": [
    "[subject:Mask R-CNN, Part-Of, Method]"
  ]
} {
  "triplets":

Saved output to: ../Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json

Question: Which method is used for instance segmentation?



Processing questions: 100%|██████████| 54/54 [1:25:55<00:00, 95.47s/it] 

Raw Assistant Response:
 {
  "triplets": [
    "[subject:Mask R-CNN, RELATION_LABEL, object:Instance Segmentation]"
  ]
}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}
{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-CNN, Used-For, object:Instance Segmentation]"]}{"triplets": ["[subject:Mask R-C

Saved output to: ../Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json

